# Pertemuan 5 — Heuristik dan Pencarian Berinformasi
**PIF403 Pengantar Kecerdasan Buatan · Prodi Informatika Universitas Adzkia**

Pencarian tanpa informasi menyapu ke segala arah. Pencarian **berinformasi** memakai **fungsi heuristik** `h(n)`: taksiran biaya dari status `n` ke tujuan. Dengan taksiran yang baik, pencarian mengarah ke tujuan dan mengekspansi jauh lebih sedikit node.

**Setelah menyelesaikan notebook ini Anda mampu:**
1. mengimplementasikan Greedy Best-First Search dan A\*;
2. memeriksa apakah sebuah heuristik *admissible* dan *consistent*;
3. merancang heuristik sendiri dan menilai mutunya dengan eksperimen.

**Notebook ini beserta notebook pertemuan 4 adalah artefak asesmen A2 (bobot 10%).** Ketentuan dan rubrik ada pada lembar tugas pertemuan 5.

In [ ]:
NAMA = "tulis nama Anda"
NIM = "tulis NIM Anda"
print(f"Notebook P5 dikerjakan oleh: {NAMA} ({NIM})")

## 1. Menyiapkan data
Unggah `peta_sumbar.csv` dan `kota_sumbar.csv`. Kali ini kolom **lintang** dan **bujur** ikut dipakai untuk menghitung jarak garis lurus.

In [ ]:
import os

BERKAS = ["peta_sumbar.csv", "kota_sumbar.csv"]
if not all(os.path.exists(b) for b in BERKAS):
    from google.colab import files
    print("Unggah berkas:", ", ".join(BERKAS))
    files.upload()
for b in BERKAS:
    assert os.path.exists(b), f"Berkas {b} belum diunggah. Jalankan ulang sel ini."

import pandas as pd, networkx as nx, matplotlib.pyplot as plt
import heapq, math, random

jalan = pd.read_csv("peta_sumbar.csv"); kota = pd.read_csv("kota_sumbar.csv")
G = nx.Graph()
for _, r in jalan.iterrows():
    G.add_edge(r.kota_asal, r.kota_tujuan, jarak=int(r.jarak_km))
POSISI = {r.kota: (r.x_gambar, r.y_gambar) for _, r in kota.iterrows()}
KOORDINAT = {r.kota: (r.lintang, r.bujur) for _, r in kota.iterrows()}
kota.head()

In [ ]:
class Masalah:
    def __init__(self, awal): self.awal = awal
    def aksi(self, s): raise NotImplementedError
    def hasil(self, s, a): raise NotImplementedError
    def tujuan(self, s): raise NotImplementedError
    def biaya(self, s, a, s2): return 1

class MasalahRute(Masalah):
    def __init__(self, graf, awal, kota_tujuan):
        super().__init__(awal); self.graf, self.kota_tujuan = graf, kota_tujuan
    def aksi(self, s): return sorted(self.graf.neighbors(s))
    def hasil(self, s, a): return a
    def tujuan(self, s): return s == self.kota_tujuan
    def biaya(self, s, a, s2): return self.graf[s][s2]["jarak"]

class Node:
    def __init__(self, status, induk=None, aksi=None, g=0):
        self.status, self.induk, self.aksi, self.g = status, induk, aksi, g
    def jalur(self):
        n, hasil = self, []
        while n: hasil.append(n.status); n = n.induk
        return hasil[::-1]

rute = MasalahRute(G, "Padang", "Payakumbuh")

## 2. Heuristik jarak garis lurus
Untuk masalah rute, heuristik yang wajar adalah **jarak garis lurus** dari sebuah kota ke kota tujuan. Jarak antara dua titik di permukaan bumi dihitung dengan rumus haversine.

In [ ]:
def garis_lurus(kota_a, kota_b):
    """Jarak garis lurus (km) antara dua kota berdasarkan lintang dan bujur."""
    (la1, lo1), (la2, lo2) = KOORDINAT[kota_a], KOORDINAT[kota_b]
    p1, p2 = math.radians(la1), math.radians(la2)
    d_lintang, d_bujur = p2 - p1, math.radians(lo2 - lo1)
    x = math.sin(d_lintang / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(d_bujur / 2) ** 2
    return 6371 * 2 * math.asin(math.sqrt(x))

def h_lurus(s):
    return garis_lurus(s, rute.kota_tujuan)

tabel_h = pd.DataFrame({"kota": sorted(G), "h = garis lurus ke Payakumbuh (km)": [round(h_lurus(k)) for k in sorted(G)]})
print(tabel_h.sort_values(tabel_h.columns[1]).to_string(index=False))

## 3. Satu kerangka, tiga algoritma
UCS, Greedy, dan A\* sebenarnya algoritma yang sama: **best-first search**. Ketiganya selalu mengekspansi node dengan nilai fungsi evaluasi `f(n)` terkecil. Bedanya hanya pada definisi `f`.

| Algoritma | f(n) | Artinya |
|---|---|---|
| UCS | g(n) | biaya yang **sudah** dikeluarkan |
| Greedy best-first | h(n) | taksiran biaya yang **masih** diperlukan |
| A\* | g(n) + h(n) | taksiran biaya **total** jalur lewat n |

In [ ]:
class Hasil:
    def __init__(self, nama, node, urutan, dibangkitkan):
        self.nama = nama
        self.jalur = node.jalur() if node else None
        self.biaya = node.g if node else None
        self.urutan_ekspansi = urutan
        self.dibangkitkan = dibangkitkan
    def __repr__(self):
        if self.jalur is None: return f"{self.nama}: tidak ditemukan"
        teks = " → ".join(map(str, self.jalur)) if len(self.jalur) <= 8 else f"{len(self.jalur) - 1} langkah"
        return f"{self.nama}: {teks} | biaya {self.biaya} | {len(self.urutan_ekspansi)} node diekspansi"

def best_first(masalah, f, nama, jejak=False):
    """Best-first search umum. f(node) menentukan prioritas (makin kecil makin dulu)."""
    awal = Node(masalah.awal)
    hitung = 0
    frontier = [(f(awal), hitung, awal)]
    g_terbaik = {}
    urutan = []
    while frontier:
        nilai_f, _, node = heapq.heappop(frontier)
        if node.status in g_terbaik and g_terbaik[node.status] <= node.g:
            continue
        g_terbaik[node.status] = node.g
        urutan.append(node.status)
        if jejak: print(f"ekspansi → {str(node.status):16s} g = {node.g:<4} f = {nilai_f:.0f}")
        if masalah.tujuan(node.status):
            return Hasil(nama, node, urutan, hitung + 1)
        for a in masalah.aksi(node.status):
            s2 = masalah.hasil(node.status, a)
            anak = Node(s2, node, a, node.g + masalah.biaya(node.status, a, s2))
            if s2 not in g_terbaik or anak.g < g_terbaik[s2]:
                hitung += 1
                heapq.heappush(frontier, (f(anak), hitung, anak))
    return Hasil(nama, None, urutan, hitung + 1)

def ucs(masalah, jejak=False):        return best_first(masalah, lambda n: n.g, "UCS", jejak)
def greedy(masalah, h, jejak=False):  return best_first(masalah, lambda n: h(n.status), "Greedy", jejak)
def astar(masalah, h, jejak=False):   return best_first(masalah, lambda n: n.g + h(n.status), "A*", jejak)

In [ ]:
print("=== Greedy best-first ===")
h_greedy = greedy(rute, h_lurus, jejak=True); print(h_greedy)
print("\n=== A* ===")
h_astar = astar(rute, h_lurus, jejak=True); print(h_astar)
print("\n=== UCS (pembanding) ===")
h_ucs = ucs(rute); print(h_ucs)

**Baca hasilnya dengan teliti.**
- **Greedy** hanya melihat `h`. Dari Padang Panjang ia memilih Batusangkar karena garis lurusnya ke Payakumbuh lebih pendek daripada Bukittinggi. Ia tidak peduli bahwa jalan Batusangkar–Payakumbuh lebih panjang. Hasilnya cepat tetapi **tidak optimal** (143 km).
- **A\*** menimbang `g + h`. Ia menemukan rute optimal 124 km dengan hanya 4 ekspansi, sedangkan UCS perlu 10 ekspansi untuk jawaban yang sama.

In [ ]:
fig, sumbu = plt.subplots(1, 3, figsize=(16, 5.8))
for ax, h in zip(sumbu, [h_ucs, h_greedy, h_astar]):
    diekspansi = set(h.urutan_ekspansi)
    warna = ["#F2A024" if k in diekspansi else "#E6F2F1" for k in G]
    nx.draw_networkx_nodes(G, POSISI, ax=ax, node_size=900, node_color=warna, edgecolors="#12263A")
    nx.draw_networkx_labels(G, POSISI, ax=ax, font_size=6.5, labels={k: k.replace(" ", "\n") for k in G})
    nx.draw_networkx_edges(G, POSISI, ax=ax, edge_color="#8A9AA5")
    nx.draw_networkx_edges(G, POSISI, ax=ax, edgelist=list(zip(h.jalur, h.jalur[1:])), edge_color="#D1495B", width=4)
    ax.set_title(f"{h.nama}: {h.biaya} km, {len(h.urutan_ekspansi)} node diekspansi (jingga)"); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Admissible dan consistent
Jaminan optimalitas A\* bergantung pada mutu heuristik.

- **Admissible**: `h(n)` tidak pernah **melebihi** biaya sebenarnya ke tujuan, `h(n) ≤ h*(n)`. Heuristiknya optimis.
- **Consistent** (monoton): untuk setiap langkah dari `n` ke `n'` berlaku `h(n) ≤ biaya(n, n') + h(n')`. Ini ketaksamaan segitiga. Heuristik yang consistent pasti admissible.

Pada masalah kecil kita bisa memeriksanya langsung: hitung `h*` (biaya sebenarnya) dengan UCS dari setiap kota, lalu bandingkan.

In [ ]:
def periksa_heuristik(graf, kota_tujuan, h):
    baris = []
    for k in sorted(graf):
        h_bintang = ucs(MasalahRute(graf, k, kota_tujuan)).biaya
        baris.append({"kota": k, "h": round(h(k), 1), "h* (sebenarnya)": h_bintang, "admissible?": h(k) <= h_bintang + 1e-9})
    tabel = pd.DataFrame(baris)
    pelanggaran = [(u, v) for u, v in graf.edges
                   if h(u) > graf[u][v]["jarak"] + h(v) + 1e-9 or h(v) > graf[u][v]["jarak"] + h(u) + 1e-9]
    return tabel, pelanggaran

tabel, pelanggaran = periksa_heuristik(G, "Payakumbuh", h_lurus)
print(tabel.to_string(index=False))
print("\nAdmissible :", bool(tabel["admissible?"].all()))
print("Consistent :", len(pelanggaran) == 0, "| sisi yang melanggar:", pelanggaran)

### Apa yang terjadi bila heuristik terlalu pesimis?
Kita kalikan heuristik dengan sebuah faktor `w`. Untuk `w > 1` heuristik bisa melebihi biaya sebenarnya (tidak lagi admissible).

In [ ]:
baris = []
for asal, akhir in [("Padang", "Payakumbuh"), ("Pariaman", "Payakumbuh"), ("Solok", "Lubuk Basung"), ("Painan", "Payakumbuh")]:
    m = MasalahRute(G, asal, akhir)
    optimal = ucs(m).biaya
    for w in (0, 1, 2, 5):
        h = astar(m, lambda s, w=w: w * garis_lurus(s, akhir))
        baris.append({"rute": f"{asal} → {akhir}", "w": w, "biaya": h.biaya, "optimal": optimal,
                      "selisih (km)": h.biaya - optimal, "node diekspansi": len(h.urutan_ekspansi)})
print(pd.DataFrame(baris).to_string(index=False))

- `w = 0` berarti `h = 0`: A\* berubah menjadi UCS (optimal tetapi banyak ekspansi).
- `w = 1`: heuristik admissible, hasil optimal dengan ekspansi jauh lebih sedikit.
- `w` besar: A\* makin mirip Greedy. Ekspansi sedikit, tetapi solusi **bisa** tidak optimal.

Ini adalah **tukar-tambah** antara mutu solusi dan upaya pencarian. Teknik menaikkan bobot heuristik secara sengaja disebut *weighted A\**.

## 5. Merancang heuristik: studi 8-puzzle
Heuristik yang baik sering diperoleh dari **masalah yang dilonggarkan** (*relaxed problem*): hapus sebagian aturan, lalu gunakan biaya solusi masalah longgar itu sebagai taksiran.

| Aturan dilonggarkan | Heuristik yang muncul |
|---|---|
| Kotak boleh pindah ke mana saja dalam satu langkah | `h1` = jumlah kotak yang salah tempat |
| Kotak boleh pindah ke petak sebelah walaupun terisi | `h2` = total jarak Manhattan |

In [ ]:
class Puzzle8(Masalah):
    TUJUAN = (1, 2, 3, 4, 5, 6, 7, 8, 0)
    GESER = {"Atas": -3, "Bawah": 3, "Kiri": -1, "Kanan": 1}
    def __init__(self, awal): super().__init__(tuple(awal))
    def aksi(self, s):
        baris, kolom = divmod(s.index(0), 3); p = []
        if baris > 0: p.append("Atas")
        if baris < 2: p.append("Bawah")
        if kolom > 0: p.append("Kiri")
        if kolom < 2: p.append("Kanan")
        return p
    def hasil(self, s, a):
        i = s.index(0); j = i + self.GESER[a]; papan = list(s)
        papan[i], papan[j] = papan[j], papan[i]; return tuple(papan)
    def tujuan(self, s): return s == self.TUJUAN

def h1(s):
    """Jumlah kotak (selain kosong) yang tidak berada di posisi tujuan."""
    return sum(1 for i, x in enumerate(s) if x != 0 and x != Puzzle8.TUJUAN[i])

def h2(s):
    """Total jarak Manhattan setiap kotak ke posisi tujuannya."""
    total = 0
    for i, x in enumerate(s):
        if x == 0: continue
        j = Puzzle8.TUJUAN.index(x)
        total += abs(i // 3 - j // 3) + abs(i % 3 - j % 3)
    return total

contoh = (7, 2, 4, 5, 0, 6, 8, 3, 1)
print("h1 =", h1(contoh), "| h2 =", h2(contoh))
for nama, h in (("h1", h1), ("h2", h2)):
    r = astar(Puzzle8(contoh), h)
    print(f"A* dengan {nama}: solusi {r.biaya} langkah, {len(r.urutan_ekspansi)} node diekspansi")

Kedua heuristik memberi solusi dengan panjang yang sama (keduanya admissible), tetapi `h2` jauh lebih hemat. Karena `h2(n) ≥ h1(n)` untuk semua `n` dan keduanya admissible, dikatakan **h2 mendominasi h1**. Heuristik yang lebih dekat ke biaya sebenarnya memangkas lebih banyak pencarian.

In [ ]:
def acak_puzzle(langkah, seed):
    rng = random.Random(seed); p = Puzzle8(Puzzle8.TUJUAN); s = p.awal; sebelum = None
    for _ in range(langkah):
        pilihan = [x for x in (p.hasil(s, a) for a in p.aksi(s)) if x != sebelum]
        sebelum, s = s, rng.choice(pilihan)
    return s

catatan = []
for seed in range(26):
    p = Puzzle8(acak_puzzle(4 + seed, seed))
    r0 = ucs(p); r1 = astar(p, h1); r2 = astar(p, h2)
    assert r0.biaya == r1.biaya == r2.biaya            # ketiganya harus optimal
    catatan.append({"d": r0.biaya, "UCS (h = 0)": len(r0.urutan_ekspansi),
                    "A* h1": len(r1.urutan_ekspansi), "A* h2": len(r2.urutan_ekspansi)})
ringkas = pd.DataFrame(catatan).groupby("d").mean().round(0).astype(int)
ringkas.index.name = "kedalaman solusi d"
print(ringkas.to_string())

In [ ]:
ax = ringkas.plot(marker="o", figsize=(8, 4.2), color=["#8A9AA5", "#F2A024", "#0E7C86"], logy=True)
ax.set_ylabel("rata-rata node diekspansi (skala log)"); ax.set_title("8-puzzle: heuristik yang lebih baik, pencarian yang lebih hemat")
ax.grid(alpha=0.3); plt.show()

---
## Tugas A2 — Notebook Pencarian dan Formulasi Masalah

Kerjakan pada sel-sel di bawah ini. Rincian ketentuan dan rubrik ada pada lembar tugas pertemuan 5.

### Bagian A — Formulasi (25%)
Buat **peta Anda sendiri**: minimal 10 simpul dan 15 sisi berbobot (contoh: kampus dan sekitarnya, kecamatan di daerah asal, jaringan trayek angkutan). Simpan sebagai dua berkas CSV dengan format kolom yang sama seperti `peta_sumbar.csv` dan `kota_sumbar.csv`, lalu **unggah** pada sel berikut. Tuliskan formulasi masalahnya (status, status awal, aksi, model transisi, uji tujuan, biaya jalur) dan sumber data jaraknya.

In [ ]:
# Bagian A: unggah peta buatan Anda
# from google.colab import files; files.upload()
# jalan_saya = pd.read_csv("peta_saya.csv"); titik_saya = pd.read_csv("titik_saya.csv")

*Formulasi masalah dan sumber data:*

### Bagian B — Algoritma (30%)
Jalankan BFS, DFS, UCS (dari notebook pertemuan 4), Greedy, dan A\* pada **tiga pasang** asal–tujuan di peta Anda. Tampilkan jalur, biaya, dan jumlah node yang diekspansi dalam satu tabel.

In [ ]:
# Bagian B

### Bagian C — Heuristik dan analisis (25%)
1. Rancang **dua** heuristik untuk peta Anda (misalnya garis lurus dan satu rancangan lain). Periksa keduanya dengan `periksa_heuristik`.
2. Tunjukkan dengan angka heuristik mana yang lebih hemat ekspansi, dan jelaskan sebabnya.
3. Buat satu heuristik yang **sengaja tidak admissible**, lalu tunjukkan satu kasus ketika A\* gagal optimal karenanya.

In [ ]:
# Bagian C

*Analisis hasil (minimal tiga paragraf: perbandingan algoritma, mutu heuristik, keterbatasan model peta Anda):*

### Bagian D — Dokumentasi (20%)
Pastikan notebook dapat dijalankan ulang dari awal tanpa galat (*Runtime → Restart and run all*), setiap bagian diberi penjelasan singkat, dan sumber data ditulis jelas.

---
**Pengumpulan A2:** unggah ke LMS `P04_NIM_Nama.ipynb`, `P05_NIM_Nama.ipynb`, serta kedua berkas CSV peta buatan Anda.